# Forecasting pipeline


In [ ]:
import os
import re
import time
import configparser
import datetime
import locale
import mimetypes
import pytz
import dateutil.tz
import json
import io
from io import StringIO, BytesIO
from datetime import date, timedelta, timezone, datetime
import calendar
from dateutil.relativedelta import relativedelta
import numpy as np
import pandas as pd
import sqlalchemy as sal
from sqlalchemy import create_engine
import urllib
import glob
from os import listdir
from os.path import isfile, join
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import sys
import warnings
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.serialize import model_to_json, model_from_json
import optuna
import itertools
import random
import pylab as pl
from sklearn.feature_selection import RFECV
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import chi2_contingency
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import shap
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyRegressor
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, VotingClassifier, VotingRegressor
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error, mean_squared_error, r2_score, make_scorer, classification_report, roc_curve, auc
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler, OneHotEncoder, PolynomialFeatures, scale
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegressionCV, LogisticRegression
from sklearn.svm import SVC
from sklearn import svm, metrics, tree
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.multiclass import OneVsRestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample, shuffle
from collections import Counter
from pandas.api.types import is_numeric_dtype
import holidays
import warnings
from scipy.stats.mstats import mquantiles
from scipy.stats import skew
from scipy.stats import randint, uniform
import lightgbm as lgb
import logging
starting_time = time.perf_counter()
print('')
print('Sanitized business label 1', datetime.now().strftime('%H:%M:%S'))
print('')
warnings.filterwarnings('ignore')
df_raw = pd.read_excel('/forecast_output_001.xlsx')
ending_time = time.perf_counter()
total_in_sec = ending_time - starting_time
print('Sanitized business label 2')
print(total_in_sec)
print('Sanitized business label 3')


# Forecasting helpers


In [ ]:
def get_holidays(country_list, year):
    """Sanitized business label 4"""
    all_holidays = {}
    if isinstance(country_list, list):
        for country in country_list:
            all_holidays.update(holidays.CountryHoliday(country, years=year).items())
    elif country_list:
        all_holidays.update(holidays.CountryHoliday(country_list, years=year).items())
    return all_holidays


In [ ]:
def process_language_data(df_selected):
    """Sanitized business label 5"""
    FC_Languages_x = list(df_selected['Language'].unique())
    processed_list = []
    for l in FC_Languages_x:
        print(f'Sanitized business label 6{l}')
        df = df_selected[df_selected['Language'] == l].copy()
        if df.empty:
            print(f'Sanitized business label 7{l}Sanitized business label 8')
            continue
        d = df['Business_Category_001'].unique()[0]
        k = df['Business_Category_002'].unique()[0]
        c = df['Country'].unique()[0]
        df['Date'] = pd.to_datetime(df['Date'])
        full_date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max())
        df_full = pd.DataFrame({'Date': full_date_range})
        df_full = df_full.merge(df, on='Date', how='left')
        df_full['Language'] = l
        df_full['Business_Category_001'] = d
        df_full['Country'] = c
        df_full['Business_Category_002'] = k
        df_full['Business_Category_003'] = df_full['Business_Category_003'].fillna(0)
        df_full = df_full.reset_index(drop=True)
        processed_list.append(df_full)
    if processed_list:
        final_df = pd.concat(processed_list, ignore_index=True)
    else:
        final_df = pd.DataFrame()
    return final_df


In [ ]:
def find_non_operating_days(df, volume_col='Business_Category_003', threshold=0.9):
    """Sanitized business label 9"""
    df['year'] = df['Date'].dt.year
    df['forecast_field_001'] = df['Date'].dt.dayofweek
    weeks_per_year = df.groupby('year')['Date'].nunique() // 7
    zero_volume_counts = df[(df[volume_col] == 0) & df['Business_Category_004'].isnull()].groupby(['year', 'forecast_field_001']).size()
    non_operating_days = []
    for (year, day), count in zero_volume_counts.items():
        if count >= weeks_per_year[year] * threshold:
            non_operating_days.append(day)
    print(f'Sanitized business label 10{sorted(set(non_operating_days))}')
    return sorted(set(non_operating_days))


In [ ]:
def optimize_prophet_with_optuna(df_volume, holidays_df, n_trials=50, initial_ratio=0.8, period_ratio=0.5):
    min_date, max_date = (df_volume['ds'].min(), df_volume['ds'].max())
    total_days = (max_date - min_date).days
    if total_days < 30:
        print(f'Sanitized business label 11{total_days}Sanitized business label 12')
        return (None, None, None)
    if total_days >= 730:
        horizon_days = 90
    elif total_days >= 365:
        horizon_days = 60
    elif total_days >= 60:
        horizon_days = 30
    else:
        horizon_days = 10
    if total_days >= 750:
        min_initial = 90
    elif total_days >= 365:
        min_initial = 60
    elif total_days >= 60:
        min_initial = 30
    else:
        min_initial = 10
    initial_days = max(int(total_days * initial_ratio), min_initial)
    remaining_days = total_days - initial_days
    if remaining_days < horizon_days:
        horizon_days = max(min(remaining_days, horizon_days), 7)
        print(f'Sanitized business label 13{horizon_days}Sanitized business label 14')
    if remaining_days < 7:
        initial_days = max(total_days - 7, min_initial)
        print(f'Sanitized business label 15{initial_days}Sanitized business label 16')
    initial = f'{initial_days} days'
    horizon = f'{horizon_days} days'
    period = f'{max(int(horizon_days * period_ratio), 7)} days'

    def objective(trial):
        try:
            params = {'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']), 'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.01, 0.1, step=0.02), 'seasonality_prior_scale': trial.suggest_categorical('seasonality_prior_scale', [5.0, 10.0]), 'holidays_prior_scale': trial.suggest_categorical('holidays_prior_scale', [5.0, 10.0]), 'fourier_order': trial.suggest_int('fourier_order', 5, 7), 'weekly_fourier_order': trial.suggest_categorical('weekly_fourier_order', [3, 5, 7]), 'weekly_prior_scale': trial.suggest_categorical('weekly_prior_scale', [0.1, 0.5, 1.0]), 'regressor_mode': trial.suggest_categorical('regressor_mode', ['additive', 'multiplicative'])}
            m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, seasonality_mode=params['seasonality_mode'], changepoint_prior_scale=params['changepoint_prior_scale'], changepoint_range=0.9, seasonality_prior_scale=params['seasonality_prior_scale'], holidays_prior_scale=params['holidays_prior_scale'], growth='logistic', interval_width=0.8, holidays=holidays_df)
            m.add_seasonality(name='monthly', period=30.5, fourier_order=params['fourier_order'])
            m.add_seasonality(name='weekly_custom', period=7, fourier_order=params['weekly_fourier_order'], prior_scale=params['weekly_prior_scale'])
            for reg in ['forecast_field_002', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006']:
                m.add_regressor(reg, mode=params['regressor_mode'])
            m.fit(df_volume)
            df_cv = cross_validation(m, initial=initial, period=period, horizon=horizon, parallel='processes')
            df_p = performance_metrics(df_cv)
            return df_p['rmse'].mean()
        except Exception as e:
            print(f'Sanitized business label 17{e}')
            return float('inf')
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    best_trial = study.best_trial
    print(f'Sanitized business label 18{best_trial.value:.4f}')
    print('Sanitized business label 19')
    for k, v in best_trial.params.items():
        print(f'  {k}: {v}')
    best_params = best_trial.params
    final_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, seasonality_mode=best_params['seasonality_mode'], changepoint_prior_scale=best_params['changepoint_prior_scale'], changepoint_range=0.9, seasonality_prior_scale=best_params['seasonality_prior_scale'], holidays_prior_scale=best_params['holidays_prior_scale'], growth='logistic', interval_width=0.8, holidays=holidays_df)
    final_model.add_seasonality(name='monthly', period=30.5, fourier_order=best_params['fourier_order'])
    final_model.add_seasonality(name='weekly_custom', period=7, fourier_order=best_params['weekly_fourier_order'], prior_scale=best_params['weekly_prior_scale'])
    for reg in ['forecast_field_002', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006']:
        final_model.add_regressor(reg, mode=best_params['regressor_mode'])
    final_model.fit(df_volume)
    final_cv = cross_validation(final_model, initial=initial, period=period, horizon=horizon, parallel='processes')
    final_perf = performance_metrics(final_cv)
    return (final_model, final_perf, best_params)


# Forecast execution


In [ ]:
def forecast_pipeline(df_raw, train_date_threshold, output_path='forecast_field_003'):
    forecast_list_volume = []
    forecast_list_kpi = []
    performance_list_volume = []
    performance_list_kpi = []
    params_list_volume = []
    params_list_kpi = []
    Train_Date_threshold = train_date_threshold
    threshold_date = datetime.strptime(Train_Date_threshold, '%Y-%m-%d')
    FC_Period = 0
    for i in range(3):
        temp_date = threshold_date + relativedelta(months=i)
        FC_Period += calendar.monthrange(temp_date.year, temp_date.month)[1]
    df_validation = df_raw[df_raw['Date'] >= Train_Date_threshold].reset_index(drop=True)
    df_selected = df_raw[df_raw['Date'] < Train_Date_threshold].reset_index(drop=True)
    df_selected_fc = df_selected.copy()
    fc_itter = df_selected_fc[['Business_Category_001', 'Business_Category_002', 'Language']].drop_duplicates(keep='first')
    fc_itter = fc_itter.sort_values(by=['Business_Category_001', 'Business_Category_002', 'Language']).reset_index(drop=True)
    for _, row in fc_itter.iterrows():
        FC_Business_Category_001 = row['Business_Category_001']
        FC_Business_Category_002 = row['Business_Category_002']
        FC_Language = row['Language']
        df_selected = df_selected_fc[(df_selected_fc['Business_Category_001'] == FC_Business_Category_001) & (df_selected_fc['Business_Category_002'] == FC_Business_Category_002) & (df_selected_fc['Language'] == FC_Language)]
        FC_Business_Category_007 = df_selected['Business_Category_007'].max()
        start_date = df_selected['Date'].min()
        end_date = df_selected['Date'].max() + pd.DateOffset(days=FC_Period)
        years = list(range(start_date.year, end_date.year + 1))
        final_df = process_language_data(df_selected)
        final_columns_list = ['Date', 'Language', 'Country', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006', 'Business_Category_003', 'Business_Category_008']
        df = final_df[final_columns_list].copy()
        non_operating_days = find_non_operating_days(df, volume_col='Business_Category_003', threshold=0.9)
        excluded_dates = df[(df['Business_Category_003'] == 0) | df['Business_Category_005'].isnull() | df['Date'].dt.dayofweek.isin(non_operating_days)]['Date'].to_list()
        df = df[~df['Date'].isin(excluded_dates)]
        df_selected_future = df_raw[(df_raw['Business_Category_001'] == FC_Business_Category_001) & (df_raw['Business_Category_002'] == FC_Business_Category_002) & (df_raw['Language'] == FC_Language)]
        df_selected_future = df_selected_future[['Date', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006']]
        df_selected_future = df_selected_future.rename(columns={'Date': 'ds'})
        FC_Country = list(df['Country'].unique())
        all_holidays_list = []
        for country in FC_Country:
            country_holidays = get_holidays(country, years)
            all_holidays_list.extend([(date, name, country) for date, name in country_holidays.items()])
        holidays_df = pd.DataFrame(all_holidays_list, columns=['ds', 'holiday', 'Country'])
        holidays_df['ds'] = pd.to_datetime(holidays_df['ds'])
        holidays_df['lower_window'] = 0
        holidays_df['upper_window'] = 1
        df['forecast_field_002'] = df['Date'].isin(holidays_df[holidays_df['Country'].isin(FC_Country)]['ds']).astype(int)
        df_prophet = df[['Date', 'forecast_field_002', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006', 'Business_Category_003', 'Business_Category_008']]
        df_prophet = df_prophet.rename(columns={'Date': 'ds', 'Business_Category_008': 'y'})
        df_volume = df_prophet[['ds', 'forecast_field_002', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006', 'Business_Category_003']].copy()
        df_volume.rename(columns={'Business_Category_003': 'y'}, inplace=True)
        cap_value_volume = df_volume['y'].max()
        floor_value_volume = df_volume[df_volume['y'] > 0]['y'].min()
        df_volume['cap'] = cap_value_volume
        df_volume['floor'] = floor_value_volume
        n_trials_n = 50
        best_model_volume, best_perf_volume, best_param_volumes = optimize_prophet_with_optuna(df_volume=df_volume, holidays_df=holidays_df, n_trials=n_trials_n)
        future_volume = best_model_volume.make_future_dataframe(periods=FC_Period, freq='D')
        future_volume = future_volume[~future_volume['ds'].isin(excluded_dates)]
        future_volume = future_volume[~future_volume['ds'].dt.dayofweek.isin(non_operating_days)]
        future_volume['forecast_field_002'] = future_volume['ds'].isin(holidays_df['ds']).astype(int)
        future_volume = future_volume.merge(df_selected_future, on='ds', how='left')
        future_volume['cap'] = cap_value_volume
        future_volume['floor'] = floor_value_volume
        future_volume = future_volume[future_volume['Business_Category_005'].isna() == False]
        volume_forecast = best_model_volume.predict(future_volume)
        df_kpi = df_prophet[['ds', 'forecast_field_002', 'Business_Category_005', 'Business_Category_004', 'Business_Category_006', 'y', 'Business_Category_003']].copy()
        cap_value = df_kpi['y'].max()
        floor_value = df_kpi[df_kpi['Business_Category_003'] > 0]['y'].min()
        del df_kpi['Business_Category_003']
        df_kpi['cap'] = cap_value
        df_kpi['floor'] = floor_value
        best_model_kpi, best_perf_kpi, best_params_kpi = optimize_prophet_with_optuna(df_volume=df_kpi, holidays_df=holidays_df, n_trials=n_trials_n)
        future_kpi = best_model_kpi.make_future_dataframe(periods=FC_Period, freq='D')
        future_kpi = future_kpi[~future_kpi['ds'].isin(excluded_dates)]
        future_kpi = future_kpi[~future_kpi['ds'].dt.dayofweek.isin(non_operating_days)]
        future_kpi['forecast_field_002'] = future_kpi['ds'].isin(holidays_df['ds']).astype(int)
        future_kpi = future_kpi.merge(df_selected_future, on='ds', how='left')
        future_kpi['cap'] = cap_value
        future_kpi['floor'] = floor_value
        future_kpi = future_kpi[future_kpi['Business_Category_005'].isna() == False]
        kpi_forecast = best_model_kpi.predict(future_kpi)
        volume_forecast['Business_Category_001'] = FC_Business_Category_001
        volume_forecast['Business_Category_002'] = FC_Business_Category_002
        volume_forecast['Language'] = FC_Language
        forecast_list_volume.append(volume_forecast)
        perf_vol = best_perf_volume.copy()
        perf_vol['Business_Category_001'] = FC_Business_Category_001
        perf_vol['Business_Category_002'] = FC_Business_Category_002
        perf_vol['Language'] = FC_Language
        performance_list_volume.append(perf_vol)
        params_list_volume.append({'Business_Category_001': FC_Business_Category_001, 'Business_Category_002': FC_Business_Category_002, 'Language': FC_Language, **best_param_volumes})
        kpi_forecast['Business_Category_001'] = FC_Business_Category_001
        kpi_forecast['Business_Category_002'] = FC_Business_Category_002
        kpi_forecast['Language'] = FC_Language
        forecast_list_kpi.append(kpi_forecast)
        perf_kpi = best_perf_kpi.copy()
        perf_kpi['Business_Category_001'] = FC_Business_Category_001
        perf_kpi['Business_Category_002'] = FC_Business_Category_002
        perf_kpi['Language'] = FC_Language
        performance_list_kpi.append(perf_kpi)
        params_list_kpi.append({'Business_Category_001': FC_Business_Category_001, 'Business_Category_002': FC_Business_Category_002, 'Language': FC_Language, **best_params_kpi})
    df_forecast_volume = pd.concat(forecast_list_volume, ignore_index=True)
    df_forecast_kpi = pd.concat(forecast_list_kpi, ignore_index=True)
    df_perf_volume = pd.concat(performance_list_volume, ignore_index=True)
    df_perf_kpi = pd.concat(performance_list_kpi, ignore_index=True)
    df_params_volume = pd.DataFrame(params_list_volume)
    df_params_kpi = pd.DataFrame(params_list_kpi)
    os.makedirs(output_path, exist_ok=True)
    df_forecast_volume.to_csv(f'{output_path}/forecast_output_002.csv', index=False)
    df_forecast_kpi.to_csv(f'{output_path}/forecast_output_003.csv', index=False)
    df_perf_volume.to_csv(f'{output_path}/forecast_output_004.csv', index=False)
    df_perf_kpi.to_csv(f'{output_path}/forecast_output_005.csv', index=False)
    df_params_volume.to_csv(f'{output_path}/forecast_output_006.csv', index=False)
    df_params_kpi.to_csv(f'{output_path}/forecast_output_007.csv', index=False)
    print(f'Sanitized business label 20{output_path}')
    return (df_forecast_volume, df_forecast_kpi, df_perf_volume, df_perf_kpi, df_params_volume, df_params_kpi)


In [ ]:
forecast_pipeline(df_raw=df_raw, train_date_threshold='2025-05-01', output_path='Business_Category_009')
